<a href="https://colab.research.google.com/github/William-HVH/Generativ-AI-with-LLMs/blob/main/Assignment_6_Generativ_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
#Document haystack dataset

#Look into the chunker - splits the documents like we saw in class
#Embedder converts it into the embeddings that we match to our query
#Retriever finds the relevant embeddings. I suspect change top-k and throwing compute at it would help a lot
##But it just does so by retrieving a ton of chunks (so of course we'll get relevant stuff)
#Reranker does the actual matching. This can be optimized.

##Generator is just for our prompt

#We put it all in our rag pipeline

#Maybe we should just uses evaluate rag to evaluate and not have the dataset at all?
##But he still downloads the goldman sachs dataset, idk

##It becomes way worse using LLM generation
##But we need to improve on the baseline performance, which I assume means no LLM

##The task is to improve on the BASELINE - which was about 85%
#Specifically: Mode: (retrieval-only)
#Accuracy: 85.45% (141/165 correct)
#Time: 8.72s (53ms per query)

# Initialization

In [4]:
!pip install -q huggingface_hub pypdf langchain-community sentence-transformers transformers accelerate
#Only needs to run this upon initialization

In [5]:
#Libraries and such here:
###OBS CHECK IF ALL OF THESE ARE NECESSARY, JUST COPY PASTED FROM NOTEBOOK FOR NOW
import os
import pandas as pd
from huggingface_hub import hf_hub_download
from pathlib import Path
import re
import time
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, util, CrossEncoder
import torch
from transformers import pipeline
##Added these
from google.colab import userdata
from huggingface_hub import login
from wandb import login as login_wandb

In [6]:
token = userdata.get("HF_token")  # get token from secret key
wandb_token = userdata.get("Wandb_token")
login(token=token) #Login hf
login_wandb(key = wandb_token)

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: williamhh (williamhh-aarhus-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Check if wandb is in use here

# Functions from notebook

In [7]:
#Taken directly from the notebook to download documents
def download_documents(document_name="GoldmanSachs", cache_dir="./haystack_data"):
    """
    Download PDFs and metadata from the Document Haystack dataset.

    Args:
        document_name: Name of document to download (e.g., "GoldmanSachs", "AIG", "AmericanAirlines")
                      Or "all" to download all available documents
        cache_dir: Local directory to store downloaded files

    Returns:
        Path to the base directory containing downloaded documents
    """
    # Available documents in the dataset
    all_documents = [
        "GoldmanSachs", "AIG", "AmericanAirlines", "APA", "BankOfMontreal",
        "BristolMyers", "CVS", "Chevron", "Cigna", "Chubb", "Comcast",
        "ConocoPhillips", "Disney", "ExxonMobil", "FedEx", "Ford",
        "GeneralMotors", "HCA", "JPMorgan", "JohnsonJohnson", "Lowes",
        "MetLife", "Progressive", "Tesla", "UnitedHealth"
    ]

    if document_name == "all":
        documents_to_download = all_documents
    else:
        if document_name not in all_documents:
            print(f"Warning: {document_name} not in known documents. Attempting anyway...")
        documents_to_download = [document_name]

    page_lengths = [5, 10, 25, 50, 75, 100, 150, 200]
    base_path = Path(cache_dir)
    base_path.mkdir(exist_ok=True)

    print(f"Downloading documents: {', '.join(documents_to_download)}")
    print("="*70)

    for doc_name in documents_to_download:
        print(f"\n📄 Downloading {doc_name}...")

        for pages in page_lengths:
            folder_name = f"{doc_name}_{pages}Pages"

            try:
                # Download PDF with text needles
                hf_hub_download(
                    repo_id="AmazonScience/document-haystack",
                    repo_type="dataset",
                    filename=f"{doc_name}/{folder_name}/{doc_name}_{pages}Pages_TextNeedles.pdf",
                    local_dir=str(base_path),
                    local_dir_use_symlinks=False
                )

                # Download needles.csv
                hf_hub_download(
                    repo_id="AmazonScience/document-haystack",
                    repo_type="dataset",
                    filename=f"{doc_name}/{folder_name}/needles.csv",
                    local_dir=str(base_path),
                    local_dir_use_symlinks=False
                )

                # Download prompt_questions.txt
                hf_hub_download(
                    repo_id="AmazonScience/document-haystack",
                    repo_type="dataset",
                    filename=f"{doc_name}/{folder_name}/prompt_questions.txt",
                    local_dir=str(base_path),
                    local_dir_use_symlinks=False
                )

            except Exception as e:
                print(f"   ✗ Error downloading {pages}-page document: {e}")

        print(f"   ✓ {doc_name} downloaded")

    return base_path

In [8]:
base_path = download_documents("GoldmanSachs") #This is just for the example - I assume it can be deleted


📄 Downloading GoldmanSachs...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


GoldmanSachs/GoldmanSachs_5Pages/Goldman(…):   0%|          | 0.00/7.26M [00:00<?, ?B/s]

needles.csv:   0%|          | 0.00/171 [00:00<?, ?B/s]

prompt_questions.txt:   0%|          | 0.00/221 [00:00<?, ?B/s]

...

prompt_questions.txt: 0.00B [00:00, ?B/s]

GoldmanSachs/GoldmanSachs_200Pages/Goldm(…):   0%|          | 0.00/7.26M [00:00<?, ?B/s]

needles.csv:   0%|          | 0.00/879 [00:00<?, ?B/s]

prompt_questions.txt: 0.00B [00:00, ?B/s]

   ✓ GoldmanSachs downloaded


In [9]:
def load_test_cases(base_path, document_name="GoldmanSachs"):
    """
    Load test cases from downloaded documents.
    """
    test_cases = []
    page_lengths = [5, 10, 25, 50, 75, 100, 150, 200]

    print(f"Looking for documents in: {base_path}")

    # The files are in base_path/DocumentName/DocumentName_XPages/

    doc_base = base_path / document_name

    if not doc_base.exists():
        print(f"ERROR: Document folder not found at {doc_base}")
        print(f"Available folders: {list(base_path.iterdir())}")
        return test_cases

    print(f"\nProcessing {document_name}...")

    for pages in page_lengths:
        folder_name = f"{document_name}_{pages}Pages"
        folder_path = doc_base / folder_name

        if not folder_path.exists():
            continue

        pdf_path = folder_path / f"{document_name}_{pages}Pages_TextNeedles.pdf"
        needles_csv_path = folder_path / "needles.csv"
        prompts_path = folder_path / "prompt_questions.txt"

        if not pdf_path.exists() or not needles_csv_path.exists():
            print(f"  ✗ Missing files in {folder_path}")
            continue

        print(f"  ✓ Loading {pages}-page document...")

        # Load PDF
        loader = PyPDFLoader(str(pdf_path))
        docs = loader.load()
        full_document = "\n\n".join([doc.page_content for doc in docs])

        # Read needles and prompts
        needles_df = pd.read_csv(needles_csv_path, header=None, names=["needle_text"])
        with open(prompts_path, 'r') as f:
            prompts = [line.strip() for line in f.readlines() if line.strip()]

        # Extract expected answers
        for idx, needle in enumerate(needles_df["needle_text"]):
            match = re.search(r'The secret (.+?) is ["\']?(.+?)["\']?\.?$', needle)
            if match and idx < len(prompts):
                key = match.group(1)
                value = match.group(2).strip('."\'')

                test_cases.append({
                    "document_name": document_name,
                    "document_length": pages,
                    "needle": needle,
                    "key": key,
                    "expected_value": value,
                    "prompt": prompts[idx],
                    "full_document": full_document
                })

        print(f"    Added {len(needles_df)} test cases")

    print(f"\n✓ Total test cases loaded: {len(test_cases)}")
    return test_cases

In [10]:
##This is just examples - I assume it can be deleted
test_cases = load_test_cases(base_path, "GoldmanSachs")
print(f"Loaded {len(test_cases)} test cases")  # Output: 165 test cases (5+10+25+25+...)

# Load from all documents
test_cases = load_test_cases(base_path, "all")

#Just to try, delete later

Looking for documents in: haystack_data

Processing GoldmanSachs...
  ✓ Loading 5-page document...
    Added 5 test cases
  ✓ Loading 10-page document...
    Added 10 test cases
  ✓ Loading 25-page document...
    Added 25 test cases
  ✓ Loading 50-page document...
    Added 25 test cases
  ✓ Loading 75-page document...
    Added 25 test cases
  ✓ Loading 100-page document...
    Added 25 test cases
  ✓ Loading 150-page document...
    Added 25 test cases
  ✓ Loading 200-page document...
    Added 25 test cases

✓ Total test cases loaded: 165
Loaded 165 test cases
Looking for documents in: haystack_data
ERROR: Document folder not found at haystack_data/all
Available folders: [PosixPath('haystack_data/.cache'), PosixPath('haystack_data/GoldmanSachs')]


Now for the building blocks

In [11]:
class Chunker:
    """Handles document chunking - easily swappable"""

    def __init__(self, chunk_size=500, chunk_overlap=100):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap
        )

    def chunk(self, document_text):
        """Split document into chunks"""
        return self.splitter.split_text(document_text)

In [12]:
class Embedder:
    """Handles embedding - easily swappable"""

    def __init__(self, model_name="BAAI/bge-small-en-v1.5"):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed(self, texts):
        """Embed texts into vectors"""
        return self.model.encode(texts, convert_to_tensor=True)

In [13]:
class Retriever:
    """Handles retrieval - easily swappable"""

    def __init__(self, embedder):
        self.embedder = embedder

    def retrieve(self, query, chunks, chunk_embeddings, top_k=5):
        """Retrieve top-k most relevant chunks"""
        query_embedding = self.embedder.embed(query)
        similarities = util.pytorch_cos_sim(query_embedding, chunk_embeddings)

        # Handle case where top_k is larger than number of chunks
        actual_k = min(top_k, len(chunks))
        top_k_indices = similarities[0].topk(actual_k).indices
        return [chunks[i] for i in top_k_indices]

In [14]:
class Reranker:
    """Handles reranking of retrieved chunks - optional component"""

    def __init__(self, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2'):
        """
        Initialize reranker with a cross-encoder model.

        Args:
            model_name: HuggingFace model name for cross-encoder
                       Popular options:
                       - 'cross-encoder/ms-marco-MiniLM-L-6-v2' (fast, good)
                       - 'BAAI/bge-reranker-base' (high quality, decent size)
        """
        self.model_name = model_name
        self.model = CrossEncoder(model_name)

    def rerank(self, query, chunks, top_k=None):
        """
        Rerank chunks based on query-chunk relevance scores.

        Args:
            query: Search query
            chunks: List of text chunks to rerank
            top_k: Return only top_k after reranking (None = return all)

        Returns:
            List of reranked chunks
        """
        # Create pairs of [query, chunk] for cross-encoder
        pairs = [[query, chunk] for chunk in chunks]

        # Get relevance scores
        scores = self.model.predict(pairs)

        # Sort chunks by score (descending)
        ranked_indices = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)
        reranked_chunks = [chunks[i] for i in ranked_indices]

        # Return top_k if specified
        if top_k:
            return reranked_chunks[:top_k]
        return reranked_chunks


In [15]:
class Generator:
    """Handles answer generation - easily swappable"""

    def __init__(self, model_name="HuggingFaceTB/SmolLM-135M-Instruct"):
        device = 0 if torch.cuda.is_available() else -1
        self.pipeline = pipeline(
            "text-generation",
            model=model_name,
            device=device
        )
        self.system_prompt = (
            "You are a helpful assistant that answers questions based on the given context. "
            "Provide direct, concise answers."
        )

    def generate(self, query, context_chunks):
        """Generate answer from query and context"""
        context = "\n".join(context_chunks)
        prompt = f"Context:\n{context}\n\nQuestion: {query}\nAnswer:"

        messages = [
            {"role": "system", "content": self.system_prompt},
            {"role": "user", "content": prompt},
        ]

        response = self.pipeline(messages, max_new_tokens=100)
        return response[0]["generated_text"][-1]["content"]

In [16]:
class RAGPipeline:
    """Complete RAG pipeline - compose all components"""

    def __init__(self, chunker, embedder, retriever, generator, reranker=None):
        """
        Initialize RAG pipeline.

        Args:
            chunker: Chunker instance
            embedder: Embedder instance
            retriever: Retriever instance
            generator: Generator instance
            reranker: Optional Reranker instance (None = no reranking)
        """
        self.chunker = chunker
        self.embedder = embedder
        self.retriever = retriever
        self.generator = generator
        self.reranker = reranker

    def prepare_document(self, document_text):
        """Prepare document for retrieval"""
        chunks = self.chunker.chunk(document_text)
        embeddings = self.embedder.embed(chunks)
        return chunks, embeddings

    def query(self, query, chunks, chunk_embeddings, top_k=5, rerank_top_k=None):
        """
        Run full RAG pipeline with optional reranking.

        Args:
            query: User query
            chunks: Document chunks
            chunk_embeddings: Pre-computed embeddings
            top_k: Number of chunks to retrieve initially
            rerank_top_k: If reranker is used, return this many after reranking
                         (None = return same as top_k)

        Returns:
            (answer, context_chunks) tuple
        """
        # Step 1: Initial retrieval with embeddings
        if self.reranker:
            # Retrieve more candidates for reranking, but not more than available chunks
            initial_k = min(top_k * 3, len(chunks))
            context_chunks = self.retriever.retrieve(query, chunks, chunk_embeddings, initial_k)

            # Step 2: Rerank the candidates
            final_k = rerank_top_k if rerank_top_k else top_k
            context_chunks = self.reranker.rerank(query, context_chunks, top_k=final_k)
        else:
            # No reranking - just retrieve
            context_chunks = self.retriever.retrieve(query, chunks, chunk_embeddings, top_k)

        # Step 3: Generate answer
        answer = self.generator.generate(query, context_chunks)

        return answer, context_chunks

# Eval:

In [17]:
# ================================================================================
# 3. EVALUATION
# ================================================================================

def evaluate_rag(test_cases, rag_pipeline, top_k=5, verbose=False, use_llm=True):
    """
    Evaluate RAG pipeline on needle-in-haystack test cases.

    Args:
        test_cases: List of test case dictionaries
        rag_pipeline: RAGPipeline instance to evaluate
        top_k: Number of chunks to retrieve
        verbose: If True, print detailed progress
        use_llm: If True, use LLM to generate answer. If False, just check if needle is in retrieved chunks.

    Returns:
        Dictionary with evaluation results
    """
    if not test_cases:
        print("ERROR: No test cases provided!")
        return {"accuracy": 0, "correct": 0, "total": 0, "time": 0, "results": []}

    results = []
    correct = 0
    total = len(test_cases)

    # Group by document for efficiency
    by_document = {}
    for case in test_cases:
        doc_key = (case["document_name"], case["document_length"])
        if doc_key not in by_document:
            by_document[doc_key] = []
        by_document[doc_key].append(case)

    start_time = time.time()

    if verbose:
        mode = "with LLM generation" if use_llm else "retrieval-only (no LLM)"
        print(f"Evaluating {total} test cases ({mode})...")
        print("="*70)

    for doc_key in sorted(by_document.keys()):
        cases = by_document[doc_key]
        doc_name, doc_length = doc_key

        if verbose:
            print(f"\n📄 {doc_name} - {doc_length} pages ({len(cases)} needles)")

        # Prepare document once
        first_case = cases[0]
        chunks, embeddings = rag_pipeline.prepare_document(first_case["full_document"])

        if verbose:
            print(f"   Chunked into {len(chunks)} chunks")

        # Test each needle
        for i, case in enumerate(cases, 1):
            expected_clean = case["expected_value"].lower().strip()
            expected_clean = expected_clean.replace('a "', '').replace('an "', '').replace('the "', '').replace('"', '').strip()

            if use_llm:
                # Use full RAG pipeline with LLM generation
                answer, context_chunks = rag_pipeline.query(
                    case["prompt"],
                    chunks,
                    embeddings,
                    top_k=top_k
                )

                # Check if expected value is in the generated answer
                answer_lower = answer.lower()
                found = expected_clean in answer_lower

            else:
                # Retrieval-only mode: just check if needle is in retrieved chunks
                if rag_pipeline.reranker:
                    # With reranker: retrieve more, then rerank
                    initial_k = min(top_k * 3, len(chunks))
                    context_chunks = rag_pipeline.retriever.retrieve(
                        case["prompt"],
                        chunks,
                        embeddings,
                        initial_k
                    )
                    context_chunks = rag_pipeline.reranker.rerank(
                        case["prompt"],
                        context_chunks,
                        top_k=top_k
                    )
                else:
                    # No reranker: just retrieve
                    context_chunks = rag_pipeline.retriever.retrieve(
                        case["prompt"],
                        chunks,
                        embeddings,
                        top_k=top_k
                    )

                # Check if expected value is in retrieved chunks
                retrieved_text = " ".join(context_chunks).lower()
                found = expected_clean in retrieved_text

                answer = "[Retrieval-only mode - no answer generated]"

            if found:
                correct += 1

            results.append({
                "document_name": doc_name,
                "document_length": doc_length,
                "prompt": case["prompt"],
                "expected": expected_clean,
                "answer": answer,
                "found": found
            })

            if verbose:
                status = "✓" if found else "✗"
                print(f"   [{i}/{len(cases)}] {status} {case['key']}: expected '{expected_clean}'")

    total_time = time.time() - start_time
    accuracy = (correct / total) * 100 if total > 0 else 0

    # Print summary
    print("\n" + "="*70)
    print("RESULTS")
    print("="*70)
    mode_str = "(with LLM)" if use_llm else "(retrieval-only)"
    print(f"Mode: {mode_str}")
    print(f"Accuracy: {accuracy:.2f}% ({correct}/{total} correct)")
    print(f"Time: {total_time:.2f}s ({total_time/total*1000:.0f}ms per query)" if total > 0 else "Time: 0.00s")
    print("="*70)

    return {
        "accuracy": accuracy,
        "correct": correct,
        "total": total,
        "time": total_time,
        "use_llm": use_llm,
        "results": results
    }


In [18]:
#I'll try this example so test cases is filled and I don't get an error:
##But I could just delete this and the test in the initialize rag below
print("Step 1: Downloading data...")
base_path = download_documents("GoldmanSachs")  # Or "AIG", "Tesla", "all", etc.

# Load test cases
print("\nStep 2: Loading test cases...")
test_cases = load_test_cases(base_path, "GoldmanSachs")
print(f"Loaded {len(test_cases)} test cases")

Step 1: Downloading data...

📄 Downloading GoldmanSachs...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(


   ✓ GoldmanSachs downloaded

Step 2: Loading test cases...
Looking for documents in: haystack_data

Processing GoldmanSachs...
  ✓ Loading 5-page document...
    Added 5 test cases
  ✓ Loading 10-page document...
    Added 10 test cases
  ✓ Loading 25-page document...
    Added 25 test cases
  ✓ Loading 50-page document...
    Added 25 test cases
  ✓ Loading 75-page document...
    Added 25 test cases
  ✓ Loading 100-page document...
    Added 25 test cases
  ✓ Loading 150-page document...
    Added 25 test cases
  ✓ Loading 200-page document...
    Added 25 test cases

✓ Total test cases loaded: 165
Loaded 165 test cases


So far, I've just copied everything from the notebook. Now I need to rework it, to get better results. Also go through and delete anything from above that is not needed

# Seems it's very important that RAGPIPELINE is defined before running this, for pipe to work correctly - OBS

In [17]:
#Use these building blocks:
# Build a basic pipeline
chunker = Chunker(chunk_size=500, chunk_overlap=100)
embedder = Embedder(model_name="BAAI/bge-small-en-v1.5")
retriever = Retriever(embedder)
generator = Generator(model_name="HuggingFaceTB/SmolLM-135M-Instruct")

# Create pipeline WITHOUT reranking
#pipeline = RAGPipeline(chunker, embedder, retriever, generator) - THIS CAUSES ISSUES AS IT OVERWRITES IMPORT PIPELINE

# Create pipeline WITH reranking
reranker = Reranker(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')
pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker) ##Renamed otherwise causes issues with overwriting pipeline that we imported

results_1 = evaluate_rag(test_cases, pipe, top_k=5, verbose=False, use_llm=False)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]


RESULTS
Mode: (retrieval-only)
Accuracy: 93.33% (154/165 correct)
Time: 8.02s (49ms per query)


# Experiment 1

The idea here is just to increase top_k - retrieve a lot more documents and bet on the reranker / embedding to be good enough to match/rank them. This should run a lot slower:

In [18]:
#Use these building blocks: - I've modified from the notebook slightly

# Build a basic pipeline
chunker = Chunker(chunk_size=500, chunk_overlap=100)
embedder = Embedder(model_name="BAAI/bge-small-en-v1.5")
retriever = Retriever(embedder)
generator = Generator(model_name="HuggingFaceTB/SmolLM-135M-Instruct") #No need here with use_llm false, but yeah

# Create pipeline WITHOUT reranking
#pipeline = RAGPipeline(chunker, embedder, retriever, generator) - THIS CAUSES ISSUES AS IT OVERWRITES IMPORT PIPELINE

# Create pipeline WITH reranking
reranker = Reranker(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')
pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker) ##Renamed otherwise causes issues with overwriting pipeline that we imported

results_1 = evaluate_rag(test_cases, pipe, top_k=100, verbose=False, use_llm=False)

Device set to use cuda:0



RESULTS
Mode: (retrieval-only)
Accuracy: 99.39% (164/165 correct)
Time: 22.91s (139ms per query)


In [25]:
embedder = Embedder(model_name="BAAI/bge-small-en-v1.5")
reranker = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")
generator = Generator(model_name="HuggingFaceTB/SmolLM-135M-Instruct")

for chunk_size in [300, 500]:
    for overlap in [50, 100]:
        chunker = Chunker(chunk_size=chunk_size, chunk_overlap=overlap)
        retriever = Retriever(embedder)
        pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker)

        print(f"\nChunk size={chunk_size}, Overlap={overlap}")

        for k in [5, 20, 40, 80]:
            print(f"  → top_k={k}")
            results = evaluate_rag(test_cases, pipe, top_k=k, verbose=False, use_llm=False)

Device set to use cuda:0



Chunk size=300, Overlap=50
  → top_k=5

RESULTS
Mode: (retrieval-only)
Accuracy: 97.58% (161/165 correct)
Time: 7.26s (44ms per query)
  → top_k=20

RESULTS
Mode: (retrieval-only)
Accuracy: 99.39% (164/165 correct)
Time: 9.52s (58ms per query)
  → top_k=40

RESULTS
Mode: (retrieval-only)
Accuracy: 99.39% (164/165 correct)
Time: 12.93s (78ms per query)
  → top_k=80

RESULTS
Mode: (retrieval-only)
Accuracy: 99.39% (164/165 correct)
Time: 18.75s (114ms per query)

Chunk size=300, Overlap=100
  → top_k=5

RESULTS
Mode: (retrieval-only)
Accuracy: 95.76% (158/165 correct)
Time: 8.28s (50ms per query)
  → top_k=20

RESULTS
Mode: (retrieval-only)
Accuracy: 98.79% (163/165 correct)
Time: 10.43s (63ms per query)
  → top_k=40

RESULTS
Mode: (retrieval-only)
Accuracy: 99.39% (164/165 correct)
Time: 13.36s (81ms per query)
  → top_k=80

RESULTS
Mode: (retrieval-only)
Accuracy: 99.39% (164/165 correct)
Time: 19.70s (119ms per query)

Chunk size=500, Overlap=50
  → top_k=5

RESULTS
Mode: (retrieval-

This keeps the exact same structure as the notebook, not changing anything from the baseline, but it varies the hyperparameters for chunk_size, overlap, and top_k - attemting to find the best combination, which I can then use for my other experiments - that build upon the LLM, embedder and reranker.
Most results were good, but the only one to yield 100% was: Chunk size=500, Overlap=50, top_k = 80.
- This of course doesn't mean that this combination is much better in practice, as a lot of queries get 164/165 correct - this was just the combination that allowed correct retrieval of the last secret for this dataset.
THIS OF COURSE - just says that we've retrieved all the right documents, not that the LLM gets them out.

# Experiment 2:

Since I already got 100%, I'll try to improve on the performance while using an LLM. I will try both with the notebooks version, and a more powerful LLM to see the difference. I will also check the results with and without reranking, to see how much of a difference it makes.
This experiments test the effects of reranking and LLM performance.
- In theory, continuing to use top_k=80 would be great, as that showed best performance before, however in practice feeding that much context to a big LLM would make it super slow. This experiment is mostly just to test the impact of reranking and quality of an LLM, hence I will downgrade the top_k to 8 (10 times less). Testing the reranker also indirectly tests the quality of our retriever, as with good retrieval, performance with/without reranking would be similar.
- It can also be assumed that the higher top_k would lead to worse performance if the LLM is not good enough to sive through the long context that it gets, and find the secret to itself.
- Hence it can be assumed that for a good LLM - high top_k would possibly mean better performance, and for a bad LLM it would mean worse performance.
  - The worse LLM improves with lower top_k, due to the reranker doing most of the heavy lifting.

Use this in argument for Qwen: https://huggingface.co/spaces/open-llm-leaderboard/open_llm_leaderboard#/

In [27]:
#Experiment 2:
#LLMs to test:
llms = [
    "HuggingFaceTB/SmolLM-135M-Instruct",
    "Qwen/Qwen2.5-3B"        #Chose Qwen 3B as the powerful model - it's the best model around 2-3B according to HF
]

rerankers = [
    None,  # no reranking
    "cross-encoder/ms-marco-MiniLM-L-6-v2"  # with reranking
]

#Now building out the test:
for llm_model in llms:
    for rer_model in rerankers:
        print(f"\nTesting LLM={llm_model}, Reranker={rer_model}")

        embedder = Embedder(model_name="BAAI/bge-small-en-v1.5")
        generator = Generator(model_name=llm_model)
        chunker = Chunker(chunk_size=500, chunk_overlap=50)
        retriever = Retriever(embedder)
        reranker = Reranker(model_name=rer_model) if rer_model else None #Only active when reranking

        pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker)
        results = evaluate_rag(test_cases, pipe, top_k=8, verbose=False, use_llm=True) #Use_llm set to true



Testing LLM=HuggingFaceTB/SmolLM-135M-Instruct, Reranker=None


Device set to use cuda:0



RESULTS
Mode: (with LLM)
Accuracy: 21.21% (35/165 correct)
Time: 465.36s (2820ms per query)

Testing LLM=HuggingFaceTB/SmolLM-135M-Instruct, Reranker=cross-encoder/ms-marco-MiniLM-L-6-v2


Device set to use cuda:0



RESULTS
Mode: (with LLM)
Accuracy: 30.30% (50/165 correct)
Time: 416.47s (2524ms per query)

Testing LLM=Qwen/Qwen2.5-3B, Reranker=None


...

Device set to use cuda:0



RESULTS
Mode: (with LLM)
Accuracy: 75.15% (124/165 correct)
Time: 471.60s (2858ms per query)

Testing LLM=Qwen/Qwen2.5-3B, Reranker=cross-encoder/ms-marco-MiniLM-L-6-v2


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0



RESULTS
Mode: (with LLM)
Accuracy: 78.18% (129/165 correct)
Time: 525.47s (3185ms per query)


As expected, the SMOLLM does better with the reranker, as the result is simply more likely to be among the context the LLM gets.
- This means that something in our embedding/retrieving process is suboptimal, and the reranker helps sort that out.

It can also be observed that using a better LLM leads to way better results. Here reranking still helps a bit, but not as much.

For the last experiment, I want to try to think about latency more. We've already gotten 100% performance without LLM,  OBS HERE CHECK and we've gotten decent performance with an LLM.
Now I want to return to not using an LLM, and try to get a high/decent result, while getting the latency as low as possible.
- For this I can keep top_k low and try other embedders/rerankers (possibly retriever? Even though the pipeline just computes it).


OBS - USE THIS FROM THE SLIDES:
"HuggingFaceTB/SmolLM-135M-Instruct": Tiny, fast, weak (default - good for testing). Check the Open LLM Leaderboard, look at overall performance (average score) and filter by size. Check the AlpacaEval Leaderboard for a quick overview of instruction-following LLMs, incl. closed models. Experiment on the prompt.

- EXPERIMENT ON THE PROMPT IN THIS LAST EXPERIMENT MAYBE. - seems like a good idea.
Maybe find a decent but not too heavy LLM, and try to run it with / without prompt and observe the difference.
- Maybe we try Qwen with reranker and then try 2/3 different prompts! :)

Use these notes for the assignment:
use_llm=False: Tests retrieval quality - Can you find the right chunks?
use_llm=True: Tests end-to-end performance - Can the LLM extract the answer
  - Use this in assignment for the different experiments.

# Experiment 3

For this experiment, I find a somewhat light instruct LLM, and test the performance on different prompt strategies.
I filtered the leaderboard to under 3B, and then checked for overall high score, and high score on IF-eval benchmark - as this deems how good the model is at following instructions.
This made me land on the model: LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct by LG AI research. (they do recommend using their own system prompt as it was trained on this but yeah)
I will test its baseline, and 2 different prompting strategies, to see how much prompting can help in RAG performance.
I will use more generic prompting strategies, as to not bias the performance towards the needle in the haystack. I want to figure out the improvement by prompting - not just adjust it to this specific dataset.

In [19]:
#I could set this up with a dictionary or whatever that would perhaps be more beautiful code, but this will do.
chunker = Chunker(chunk_size=500, chunk_overlap=50)
embedder = Embedder(model_name="BAAI/bge-small-en-v1.5")
retriever = Retriever(embedder)
reranker = Reranker(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')
generator = Generator(model_name="LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")
pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker)

#Baseline prompt (keeps original prompt)
results_baseline = evaluate_rag(test_cases, pipe, top_k=8, verbose=False, use_llm=True)
#Prompt:  "You are a helpful assistant that answers questions based on the given context. "Provide direct, concise answers."

#Perhaps better instructions from this prompt
generator.system_prompt = (
    "You are an assistant that helps figure out the precise answer, burried in a long context."
    "You will retrieve this answer, and then concisely write it out."
)
pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker) #I reset the pipe. Not sure if it's necessary, but it might keep old context in memory.
results2 = evaluate_rag(test_cases, pipe, top_k=8, verbose=False, use_llm=True)

#This prompt is a bit for fun, as the llm of course doesn't inherently understand the context - but I am trying to invoke tokens linked with urgency and expertise.
#Hopefully we shift the weights towards what the model deems to be "correctness" and "expertise"
generator.system_prompt = (
    "You are a university professor specialized in filtering through documents to find the right context."
    "You are the top expert in the world for this task."
    "Recently there has been accusations of the quality of your work by your peers."
    "To prove them wrong, you will masterfully find the correct answer through the long context given to you."
    "Your job is on the line should you fail, so getting the correct answer is very important."
)
pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker)
results3 = evaluate_rag(test_cases, pipe, top_k=8, verbose=False, use_llm=True)


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/743 [00:00<?, ?B/s]

...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/563 [00:00<?, ?B/s]

Device set to use cuda:0
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



RESULTS
Mode: (with LLM)
Accuracy: 90.30% (149/165 correct)
Time: 113.88s (690ms per query)

RESULTS
Mode: (with LLM)
Accuracy: 91.52% (151/165 correct)
Time: 113.59s (688ms per query)

RESULTS
Mode: (with LLM)
Accuracy: 92.12% (152/165 correct)
Time: 441.72s (2677ms per query)


I kept embedder/reranker the same throughout (as I had already gotten good retrieval), further experiments could look to change these.
It would also be interesting to see the performacen of this LLM with high context - i.e increasing top_k. (80 here should still be way within context window)
So I'll quickly do this - combining what we've learned about chunk_size/overlap, top_k, better LLM's for higher performance, instructions and system promts:

Let's scale this to full retrieval - top_k=80, also check the verbose just for fun :). I could also change the embedder/reranker, but for now this'll do.

In [25]:
#I could set this up with a dictionary or whatever that would perhaps be more beautiful code, but this will do.
chunker = Chunker(chunk_size=500, chunk_overlap=50)
embedder = Embedder(model_name="BAAI/bge-small-en-v1.5")
retriever = Retriever(embedder)
reranker = Reranker(model_name='cross-encoder/ms-marco-MiniLM-L-6-v2')
generator = Generator(model_name="LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct")
generator.system_prompt = (
    "You are a university professor specialized in filtering through documents to find the right context."
    "You are the top expert in the world for this task."
    "Recently there has been accusations of the quality of your work by your peers."
    "To prove them wrong, you will masterfully find the correct answer through the long context given to you."
    "Your job is on the line should you fail, so getting the correct answer is very important."
)
pipe = RAGPipeline(chunker, embedder, retriever, generator, reranker=reranker)
results = evaluate_rag(test_cases, pipe, top_k=80, verbose=True, use_llm=True)

The repository LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct .
 You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y
The repository LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct contains custom code which must be executed to correctly load the model. You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct .
 You can inspect the repository content at https://hf.co/LGAI-EXAONE/EXAONE-3.5-2.4B-Instruct.
You can avoid this prompt in future by passing the argument `trust_remote_code=True`.

Do you wish to run the custom code? [y/N] y


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0


Evaluating 165 test cases (with LLM generation)...

📄 GoldmanSachs - 5 pages (5 needles)
   Chunked into 10 chunks
   [1/5] ✓ flower: expected 'lavender'
   [2/5] ✓ tool: expected 'scissors'
   [3/5] ✓ shape: expected 'star'
   [4/5] ✓ clothing: expected 'dress'
   [5/5] ✓ office supply: expected 'envelope'

📄 GoldmanSachs - 10 pages (10 needles)
   Chunked into 23 chunks
   [1/10] ✓ flower: expected 'lavender'
   [2/10] ✓ tool: expected 'scissors'
   [3/10] ✓ shape: expected 'star'
   [4/10] ✓ clothing: expected 'dress'
   [5/10] ✓ office supply: expected 'envelope'
   [6/10] ✓ fruit: expected 'grape'
   [7/10] ✓ drink: expected 'milk'
   [8/10] ✓ transportation: expected 'airplane'
   [9/10] ✓ landmark: expected 'colosseum'
   [10/10] ✓ kitchen appliance: expected 'toaster'

📄 GoldmanSachs - 25 pages (25 needles)
   Chunked into 111 chunks
   [1/25] ✓ flower: expected 'lavender'
   [2/25] ✓ tool: expected 'scissors'
   [3/25] ✓ shape: expected 'star'
   [4/25] ✓ clothing: expected 'd

We get good performance of 97,58%, but not we cannot recreate the 100% score of pure retrieval through the LLM
